In [1]:
!pip install ucimlrepo umap-learn kagglehub --quiet


In [2]:
import numpy as np
import pandas as pd

from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

import umap
import tensorflow as tf


In [3]:
retention_levels = [0.10, 0.25, 0.50, 0.75, 0.90]


In [4]:
def umap_classification_experiment(X, y, retention_levels):
    results = []

    # Train-test split (stratified)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    # Standardization
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # ------------------
    # Baseline (no DR)
    # ------------------
    baseline_clf = KNeighborsClassifier(n_neighbors=5)
    baseline_clf.fit(X_train_scaled, y_train)
    baseline_acc = accuracy_score(
        y_test, baseline_clf.predict(X_test_scaled)
    ) * 100

    original_dims = X.shape[1]

    # ------------------
    # UMAP experiments
    # ------------------
    for r in retention_levels:
        n_components = max(2, int(original_dims * r))  # UMAP needs ≥2 dims

        reducer = umap.UMAP(
            n_components=n_components,
            n_neighbors=15,
            min_dist=0.1,
            metric='euclidean',
            random_state=42
        )

        X_train_umap = reducer.fit_transform(X_train_scaled)
        X_test_umap = reducer.transform(X_test_scaled)

        clf = KNeighborsClassifier(n_neighbors=5)
        clf.fit(X_train_umap, y_train)

        acc = accuracy_score(
            y_test, clf.predict(X_test_umap)
        ) * 100

        results.append({
            'Retention Level (%)': int(r * 100),
            'UMAP Components': n_components,
            'Accuracy (%)': acc
        })

    return baseline_acc, pd.DataFrame(results)


In [5]:
wine_data = fetch_ucirepo(id=186)  # Wine Quality
X_wine = wine_data.data.features.values
y_wine = wine_data.data.targets.values.ravel()

baseline_wine, wine_umap = umap_classification_experiment(
    X_wine, y_wine, retention_levels
)

wine_umap['Dataset'] = 'Wine'
wine_umap['Method'] = 'UMAP'
wine_umap['Baseline (%)'] = baseline_wine

wine_umap


/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


,Retention Level (%),UMAP Components,Accuracy (%),Dataset,Method,Baseline (%)
0,10,2,51.615385,Wine,UMAP,55.846154
1,25,2,51.615385,Wine,UMAP,55.846154
2,50,5,53.230769,Wine,UMAP,55.846154
3,75,8,54.461538,Wine,UMAP,55.846154
4,90,9,53.538462,Wine,UMAP,55.846154


In [6]:
breast_data = fetch_ucirepo(id=17)  # Breast Cancer Wisconsin
X_breast = breast_data.data.features.values
y_breast = breast_data.data.targets.values.ravel()

baseline_breast, breast_umap = umap_classification_experiment(
    X_breast, y_breast, retention_levels
)

breast_umap['Dataset'] = 'Breast Cancer'
breast_umap['Method'] = 'UMAP'
breast_umap['Baseline (%)'] = baseline_breast

breast_umap


/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


,Retention Level (%),UMAP Components,Accuracy (%),Dataset,Method,Baseline (%)
0,10,3,93.859649,Breast Cancer,UMAP,95.614035
1,25,7,94.736842,Breast Cancer,UMAP,95.614035
2,50,15,93.859649,Breast Cancer,UMAP,95.614035
3,75,22,92.982456,Breast Cancer,UMAP,95.614035
4,90,27,94.736842,Breast Cancer,UMAP,95.614035


In [7]:
(X_train, y_train), (_, _) = tf.keras.datasets.mnist.load_data()

X_mnist = X_train.reshape(X_train.shape[0], -1)
y_mnist = y_train

baseline_mnist, mnist_umap = umap_classification_experiment(
    X_mnist, y_mnist, retention_levels
)

mnist_umap['Dataset'] = 'MNIST'
mnist_umap['Method'] = 'UMAP'
mnist_umap['Baseline (%)'] = baseline_mnist

mnist_umap


11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


,Retention Level (%),UMAP Components,Accuracy (%),Dataset,Method,Baseline (%)
0,10,78,91.500000,MNIST,UMAP,94.783333
1,25,196,91.741667,MNIST,UMAP,94.783333
2,50,392,91.791667,MNIST,UMAP,94.783333
3,75,588,91.508333,MNIST,UMAP,94.783333
4,90,705,91.691667,MNIST,UMAP,94.783333


In [8]:
umap_table = pd.concat(
    [wine_umap, breast_umap, mnist_umap],
    ignore_index=True
)

umap_table = umap_table[
    ['Dataset', 'Method', 'Baseline (%)',
     'Retention Level (%)', 'UMAP Components', 'Accuracy (%)']
]

umap_table


,Dataset,Method,Baseline (%),Retention Level (%),UMAP Components,Accuracy (%)
0,Wine,UMAP,55.846154,10,2,51.615385
1,Wine,UMAP,55.846154,25,2,51.615385
2,Wine,UMAP,55.846154,50,5,53.230769
3,Wine,UMAP,55.846154,75,8,54.461538
4,Wine,UMAP,55.846154,90,9,53.538462
5,Breast Cancer,UMAP,95.614035,10,3,93.859649
6,Breast Cancer,UMAP,95.614035,25,7,94.736842
7,Breast Cancer,UMAP,95.614035,50,15,93.859649
8,Breast Cancer,UMAP,95.614035,75,22,92.982456
9,Breast Cancer,UMAP,95.614035,90,27,94.736842
